# Sanity check

Quick sanity queries against `raw_matches` in the DuckDB warehouse: top scoring teams, home win %, and a goals-per-season trend.

In [1]:
import sys
sys.path.insert(0, "../src")

from db import get_connection

con = get_connection()
con.execute("SELECT COUNT(*) FROM raw_matches").fetchone()

(760,)

## Top scoring teams (goals for, home + away)

In [2]:
top_scorers = con.execute("""
    SELECT team, SUM(goals) AS goals_for
    FROM (
        SELECT home_team AS team, fthg AS goals FROM raw_matches
        UNION ALL
        SELECT away_team AS team, ftag AS goals FROM raw_matches
    )
    GROUP BY team
    ORDER BY goals_for DESC
    LIMIT 10
""").df()
top_scorers

,team,goals_for
0,Man City,190.0
1,Arsenal,179.0
2,Liverpool,161.0
3,Newcastle,153.0
4,Tottenham,144.0
5,Brighton,127.0
6,Aston Villa,127.0
7,Man United,115.0
8,Chelsea,115.0
9,Brentford,114.0


## Home win %

In [3]:
home_win_pct = con.execute("""
    SELECT
        100.0 * SUM(CASE WHEN ftr = 'H' THEN 1 ELSE 0 END) / COUNT(*) AS home_win_pct
    FROM raw_matches
""").df()
home_win_pct

,home_win_pct
0,47.236842


## Goals trend by season

In [4]:
goals_trend = con.execute("""
    SELECT
        season,
        COUNT(*) AS matches,
        AVG(fthg + ftag) AS avg_goals_per_match
    FROM raw_matches
    GROUP BY season
    ORDER BY season
""").df()
goals_trend

,season,matches,avg_goals_per_match
0,2223,380,2.852632
1,2324,380,3.278947
